# Import needed packages, Models, and Data

In [ ]:
import pandas as pd
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import torch
from transformers import pipeline

# this basically means "use my GPU for machine learning shit"
device = 0 if torch.cuda.is_available() else -1

# connect to database and init a cursor for querying
xct_params = {
    "user":                 os.getenv('SF_USR')
   ,"account":              os.getenv('SF_ID')
   ,"warehouse":            os.getenv('SF_WH')
   ,"database":             os.getenv('SF_DB')
   ,"schema":               os.getenv('SF_SC')
   ,"role":                 os.getenv('SF_RL')
   ,"private_key_file":     os.getenv('PRIVATE_KEY_PATH')
   ,"private_key_file_pwd": os.getenv('PRIVATE_KEY_PASSPHRASE')
   ,"authenticator":        os.getenv('SF_AUTH')
}

SF_XCT = snowflake.connector.connect(**xct_params)

CSR = SF_XCT.cursor()

# sentiment analyzer doo-dad instantiation
PIPL_SENT = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment",
    device=device,
    truncation=True,
    max_length = 512 
)
## this sentiment model has the below mapping that indicates the overall sentiment returned
## for verification, see this link:
##      https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment

# named-entity recognition doo-dad instantiation
PIPL_NER = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    tokenizer="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=device,
    batch_size=256 
)

# capture the data what needs analyzing
sample_size_in_percent = 50
query = f"""
with src as (
select content_id
      ,usa_timestamp
      ,detected_languages
      ,post_text 
from {SF_DB}.main.firehose_processed sample system ({sample_size_in_percent})
where detected_languages = '["en"]'
)

select * 
from src
;
"""
CSR.execute(query)
DATA = CSR.fetch_pandas_all()

TypeError: Password was not given but private key is encrypted

# Apply Sentiment and NER Analysis

In [2]:
os.getenv('PRIVATE_KEY_PASSPHRASE')

'7K7yiRS0wSGnnzGUcoKc'

In [ ]:
sentiment_output = PIPL_SENT(DATA['POST_TEXT'].tolist())
ner_output       = PIPL_NER(DATA['POST_TEXT'].tolist()) 
# last runtime (15%) = 26m 14.7s 

# Write Results to Snowflake

In [4]:
DATA['SENTIMENT_ANALYSIS'] = sentiment_output
DATA['NER_ANALYSIS']       = ner_output
DATA['SAMPLING_PERCENT']   = sample_size_in_percent

write_pandas(SF_XCT, DATA
            ,table_name='NER_SENTANA_OUTPUT_SAMPLE'
            ,database=SF_DB.replace('"', '').upper()
            ,schema=SF_SC.replace('"', '').upper()
           )

/tmp/ipykernel_33957/3960979939.py:5: UserWarning: Dataframe contains a datetime with timezone column, but 'use_logical_type=None'. This can result in dateimes being incorrectly written to Snowflake. Consider setting 'use_logical_type = True'
  write_pandas(SF_XCT, DATA


(True,
 1,
 58320,
 [('agcbaygyfn/file0.txt',
   'LOADED',
   58320,
   58320,
   1,
   0,
   None,
   None,
   None,
   None)])